In [4]:
from dotenv import load_dotenv
import openai
import requests
import os
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


PROMPT = """
너는 유능한 영화 전문 AI를 만들거다.
여기서 사용할 모델은 "gpt-4o-mini" 사용하도록 한다.
영화 관련 소스는 모두 BASE_URL에서 가져오도록 한다.


이제 세가지의 함수를 만들어야한다.

get_popular_movies() - /movies에서 인기 영화를 가져옵니다.
get_movie_details(id) - /movies/:id에서 영화 정보를 가져옵니다.
get_similar_movies(id) - /movies/:id/similar에서 유사한 영화를 조회합니다.


이 세가지 함수를 꼭 사용해서 정보를 가져오도록 한다.
모델이 함수명과 인자(Arguments)를 올바르게 출력해야한다.


"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": PROMPT}],
)

message = response.choices[0].message.content
print(message)

아래는 요청하신 세 가지 함수의 구현 예시입니다. 각 함수는 BASE_URL을 사용하여 영화 정보를 가져오도록 설계되었습니다.

```python
BASE_URL = "https://api.example.com"  # 여기에 실제 API의 BASE_URL을 입력해야 합니다.

import requests

def get_popular_movies():
    """
    BASE_URL의 /movies 엔드포인트에서 인기 영화를 가져옵니다.
    """
    response = requests.get(f"{BASE_URL}/movies")
    if response.status_code == 200:
        return response.json()  # 인기 영화의 JSON 데이터를 반환합니다.
    else:
        raise Exception(f"Error fetching popular movies: {response.status_code}")

def get_movie_details(id):
    """
    BASE_URL의 /movies/:id 엔드포인트에서 특정 영화의 상세 정보를 가져옵니다.

    :param id: 영화 ID
    """
    response = requests.get(f"{BASE_URL}/movies/{id}")
    if response.status_code == 200:
        return response.json()  # 영화 상세 정보의 JSON 데이터를 반환합니다.
    else:
        raise Exception(f"Error fetching movie details for ID {id}: {response.status_code}")

def get_similar_movies(id):
    """
    BASE_URL의 /movies/:id/similar 엔드포인트에서 유사한 영화를 조회합니다.

    :param id: 영화 ID
    """
    re

In [5]:
from openai.types.chat import ChatCompletionMessage
import requests
import os
import openai
import json
from dotenv import load_dotenv

# 환경 변수를 로드합니다.
load_dotenv()
client = openai.OpenAI()
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
messages=[]
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    if response.status_code == 200:
        return response.json()  # 인기 영화 목록을 반환
    else:
        return {"error": "Unable to fetch popular movies."}

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    if response.status_code == 200:
        return response.json()  # 특정 영화의 세부정보를 반환
    else:
        return {"error": f"Unable to fetch movie details for ID: {id}"}

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    if response.status_code == 200:
        return response.json()  # 유사한 영화 목록을 반환
    else:
        return {"error": f"Unable to fetch similar movies for ID: {id}"}

In [7]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_popular_movies",
        "description": "인기 영화 목록을 가져옵니다.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }},
    {"type": "function", "function": {
        "name": "get_movie_details",
        "description": "특정 ID의 영화 상세 정보를 가져옵니다.",
        "parameters": {"type": "object", "properties": {
            "id": {"type": "integer", "description": "영화 ID"}
        }, "required": ["id"]},
    }},
    {"type": "function", "function": {
        "name": "get_similar_movies",
        "description": "특정 ID의 영화와 유사한 영화를 조회합니다.",
        "parameters": {"type": "object", "properties": {
            "id": {"type": "integer", "description": "영화 ID"}
        }, "required": ["id"]},
    }},
]


FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_similar_movies': get_similar_movies,
}

In [8]:
def _tool_result_to_str(result) -> str:
    if isinstance(result, str):
        return result
    return json.dumps(result, ensure_ascii=False)


def process_ai_response(message: ChatCompletionMessage) -> bool:
    if message.tool_calls:
        assistant_msg = {
            "role": "assistant",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
                for tool_call in message.tool_calls
            ],
        }
        if message.content:
            assistant_msg["content"] = message.content
        messages.append(assistant_msg)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            raw_args = tool_call.function.arguments

            print(f"Calling function: {function_name} with {raw_args}")

            try:
                arguments = json.loads(raw_args)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            if function_to_run is None:
                result = {"error": f"Unknown function: {function_name}"}
            else:
                result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": _tool_result_to_str(result),
                }
            )
        return True

    messages.append({"role": "assistant", "content": message.content})
    print(f"AI: {message.content}")
    return False

In [9]:
def call_ai():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message
        had_tools = process_ai_response(msg)
        if not had_tools:
            break


while True:
    message = input("Send a message to the LLM..")
    if message == "quit" or message == "q":
        break
    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    call_ai()

User: hi
AI: Hello! How can I assist you today?
User: 가장 인기있는 영화 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {}
AI: 여기 현재 가장 인기 있는 영화 목록입니다:

1. **Your Heart Will Be Broken** (Твоё сердце будет разбито)
   - **개요**: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하다가 주 알림자 바르스와 일종의 거래를 하게 됩니다.
   - **개봉일**: 2026-03-26
   - **평점**: 6.75
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - **개요**: 제이크 숄리와 네이티리는 팬도라에서 새로운 위협, 아쉬인족과 싸워야 합니다.
   - **개봉일**: 2025-12-17
   - **평점**: 7.37
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - **개요**: 인간의 의식을 리얼한 로봇 동물에 '튀게' 하는 기술로 동물들과 소통할 수 있습니다.
   - **개봉일**: 2026-03-04
   - **평점**: 7.60
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Shelter**
   - **개요**: 스스로 자발적으로 은둔한 남자가 폭풍으로부터 소녀를 구하면서 벌어지는 이야기입니다.
   - **개봉일**: 2026-01-28
   - **평점**: 6.79
   - ![포스터](https://image.tmdb.org/t/p/w780/buPFn